In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="2FEcd5GPp2Mt7psXe2TP")
project = rf.workspace("hussains-workspace-yhos3").project("car-license-plate-detection-maehj-t5jdf")
version = project.version(1)
dataset = version.download("yolov11")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 111.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Car-License-Plate--Detection-1 in yolov11:: 100%|██████████| 605/605 [00:00<00:00, 7169.71it/s]


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.2 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO
import torch

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
model = YOLO('yolov8s.pt')

In [6]:
results = model.train(
    # Dataset
    data        = '/content/Car-License-Plate--Detection-1/data.yaml',

    # Core
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,

    # Optimizer
    optimizer   = 'AdamW',
    lr0         = 0.001,              # initial learning rate
    lrf         = 0.01,               # final lr = lr0 * lrf
    momentum    = 0.937,
    weight_decay= 0.0005,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,

    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 5.0,
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    perspective = 0.0001,
    flipud      = 0.0,
    fliplr      = 0.3,
    mosaic      = 1.0,
    mixup       = 0.1,

    # Confidence & NMS
    conf        = 0.25,
    iou         = 0.7,

    # Output
    project     = 'license_plate_model',
    name        = 'train_v1',
    save        = True,
    save_period = 10,

    # Settings
    workers     = 4,
    device      = 0 if torch.cuda.is_available() else 'cpu',
    patience    = 20,                 # early stopping
    verbose     = True,
    plots       = True,
)

print("\n✅ Training Complete!")
print(f"Best weights : {results.save_dir}/weights/best.pt")


Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Car-License-Plate--Detection-1/data.yaml, degrees=5.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.3, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_v1-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, over

KeyboardInterrupt: 

In [ ]:
best_model = YOLO('license_plate_model/train_v1/weights/best.pt')

metrics = best_model.val(data='/content/Car-License-Plate--Detection-1/data.yaml')

print(f"\n📊 Validation Results:")
print(f"   mAP50     : {metrics.box.map50:.4f}")
print(f"   mAP50-95  : {metrics.box.map:.4f}")
print(f"   Precision : {metrics.box.mp:.4f}")
print(f"   Recall    : {metrics.box.mr:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════
#  TEST — Single Image pe Bounding Box Dekho
# ═══════════════════════════════════════════════════════
import cv2
import matplotlib.pyplot as plt

best_model = YOLO('license_plate_model/train_v1/weights/best.pt')

def detect_plate(image_path, conf=0.25):
    img     = cv2.imread(image_path)
    results = best_model(img, conf=conf, verbose=False)[0]

    boxes = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        confidence      = float(box.conf[0])
        boxes.append((x1, y1, x2, y2, confidence))

        # Draw
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f"plate {confidence:.2f}",
                    (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 255, 0), 2)

    # Show
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Detected Plates: {len(boxes)}")
    plt.axis('off')
    plt.show()

    return boxes   # ← sirf bounding boxes return ho rahi hain

# Test
test_image = "test_car.jpg"
boxes = detect_plate(test_image)
print(f"\n📦 Bounding Boxes: {boxes}")
# Output: [(x1, y1, x2, y2, confidence), ...]